In [1]:
!pip install torch-geometric transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import json
import glob
import numpy as np
from pathlib import Path
from collections import Counter

In [3]:
RAW_DIR   = "/kaggle/input/datasets/taanhkhoa/facebook-graph/data/raw"
MEDIA_DIR = "/kaggle/input/datasets/taanhkhoa/facebook-graph/data/media"

json_files = sorted(glob.glob(f"{RAW_DIR}/*.json"))
print(f"Found {len(json_files)} posts")

posts = [json.loads(open(f).read()) for f in json_files]
print(f"Loaded {len(posts)} posts")

Found 902 posts
Loaded 902 posts


In [4]:
n_comments = []
n_edges = []
for d in posts:
  gs = d.get("graph_structure", {})
  def count_all(cmts):
      n = len(cmts)
      for c in cmts: n += count_all(c.get("replies", []))
      return n
  nc = count_all(gs.get("comment_tree", []))
  ne = len(gs.get("edges_user_user", [])) + len(gs.get("edges_comment_reply", []))
  n_comments.append(nc)
  n_edges.append(ne)

print(f"Avg comments/post : {np.mean(n_comments):.1f}  (max {max(n_comments)})")
print(f"Avg edges/post    : {np.mean(n_edges):.1f}  (max {max(n_edges)})")
print(f"Total edges       : {sum(n_edges):,}")

Avg comments/post : 128.3  (max 1029)
Avg edges/post    : 255.8  (max 1332)
Total edges       : 230,743


In [5]:
import json
import numpy as np
from pathlib import Path
from typing import Dict, Any, Tuple
from collections import defaultdict

def json_to_hetero_dict(json_path: str) -> Dict[str, Any]:
  with open(json_path) as f:
      d = json.load(f)

  gs = d["graph_structure"]
  post_id = d["post_id"]
  eng = d.get("engagement", {})

  node_ids = {"post": {post_id: 0}, "user": {}, "comment": {}, "hashtag": {}}

  def get_id(ntype, nid):
      if nid not in node_ids[ntype]:
          node_ids[ntype][nid] = len(node_ids[ntype])
      return node_ids[ntype][nid]

  get_id("post", post_id)
  if gs.get("author_id"): get_id("user", gs["author_id"])

  def register_comments(cmts):
      for c in cmts:
          get_id("comment", c["comment_id"])
          if c.get("author_id"): get_id("user", c["author_id"])
          register_comments(c.get("replies", []))
  register_comments(gs.get("comment_tree", []))

  for h in gs.get("hashtag_nodes", []): get_id("hashtag", h["hashtag"])
  for e in gs.get("edges_user_user", []):
      get_id("user", e["source_user_id"]); get_id("user", e["target_user_id"])

  edges = defaultdict(list)

  for e in gs.get("edges_user_post", []):
      if e["interaction_type"] == "author" and e.get("user_id"):
          edges[("user","author","post")].append(
              (get_id("user", e["user_id"]), get_id("post", post_id)))

  for e in gs.get("edges_user_comment", []):
      if e.get("user_id") and e.get("comment_id"):
          edges[("user","author","comment")].append(
              (get_id("user", e["user_id"]), get_id("comment", e["comment_id"])))

  for e in gs.get("edges_user_user", []):
      src, tgt = e["source_user_id"], e["target_user_id"]
      if src and tgt:
          edges[("user", e["relation_type"], "user")].append(
              (get_id("user", src), get_id("user", tgt)))

  for e in gs.get("edges_comment_reply", []):
      cid, tid, ttype, direction = e["comment_id"], e["target_id"], e["target_type"], e["direction"]
      if not cid or not tid: continue
      if ttype == "post":
          if cid in node_ids["comment"] and tid in node_ids["post"]:
              edges[("comment", direction, "post")].append(
                  (get_id("comment", cid), get_id("post", tid)))
      else:
          if cid in node_ids["comment"] and tid in node_ids["comment"]:
              edges[("comment", direction, "comment")].append(
                  (get_id("comment", cid), get_id("comment", tid)))

  for e in gs.get("edges_post_hashtag", []):
      if e["direction"] == "has_hashtag":
          edges[("post","has_hashtag","hashtag")].append(
              (get_id("post", e["post_id"]), get_id("hashtag", e["hashtag"])))

  edge_index_dict = {
      k: np.array(v, dtype=np.int64).T
      for k, v in edges.items() if v
  }

  post_engagement = np.array([[
      eng.get("like",0), eng.get("love",0), eng.get("haha",0),
      eng.get("wow",0),  eng.get("sad",0),  eng.get("angry",0),
      eng.get("care",0), eng.get("comment_count",0),
      eng.get("share_count",0), eng.get("total_reactions",0),
  ]], dtype=np.float32)

  return {
      "post_id": post_id,
      "num_nodes": {ntype: len(ids) for ntype, ids in node_ids.items()},
      "node_ids": node_ids,
      "edge_index": edge_index_dict,
      "post_engagement": post_engagement,
      "raw_text": d.get("node_features", {}).get("text", ""),
      "image_urls": d.get("node_features", {}).get("image_urls", []),
      "local_images": d.get("node_features", {}).get("local_images", []),
  }

print("json_to_hetero_dict loaded")

json_to_hetero_dict loaded


In [6]:
import torch
from torch_geometric.data import HeteroData

def to_pyg_heterodata(json_path: str) -> HeteroData:
  data_dict = json_to_hetero_dict(json_path)
  data = HeteroData()

  FEAT_DIM = 1
  for ntype, count in data_dict["num_nodes"].items():
      data[ntype].x = torch.zeros(count, FEAT_DIM)
      data[ntype].node_id = list(data_dict["node_ids"][ntype].keys())

  for (src, rel, dst), edge_index in data_dict["edge_index"].items():
      data[src, rel, dst].edge_index = torch.from_numpy(edge_index)

  data["post"].engagement = torch.from_numpy(data_dict["post_engagement"])

  return data

print("to_pyg_heterodata loaded")

to_pyg_heterodata loaded


In [7]:
from torch_geometric.data import Dataset

class VNSocialGraphDataset(Dataset):
  def __init__(self, files):
      super().__init__()
      self.files = files

  def len(self):
      return len(self.files)

  def get(self, idx):
      return to_pyg_heterodata(self.files[idx])

dataset = VNSocialGraphDataset(json_files)
print(f"Dataset size: {len(dataset)}")
print(f"Sample: {dataset[0]}")

Dataset size: 902
Sample: HeteroData(
  post={
    x=[1, 1],
    node_id=[1],
    engagement=[1, 10],
  },
  user={
    x=[155, 1],
    node_id=[155],
  },
  comment={
    x=[186, 1],
    node_id=[186],
  },
  hashtag={
    x=[0, 1],
    node_id=[0],
  },
  (user, author, post)={ edge_index=[2, 1] },
  (user, author, comment)={ edge_index=[2, 186] },
  (user, mention, user)={ edge_index=[2, 4] },
  (user, reply, user)={ edge_index=[2, 89] },
  (user, mention_rev, user)={ edge_index=[2, 4] },
  (user, reply_rev, user)={ edge_index=[2, 89] },
  (comment, reply_to, post)={ edge_index=[2, 93] },
  (comment, reply_to, comment)={ edge_index=[2, 93] },
  (comment, reply_to_rev, comment)={ edge_index=[2, 93] }
)


In [8]:
sample = dataset[0]

print("=" * 60)
print("HETERODATA STRUCTURE")
print("=" * 60)

print("\n📦 NODE TYPES:")
for ntype in sample.node_types:
  x = sample[ntype].x
  ids = sample[ntype].node_id
  print(f"  {ntype:10} → {x.shape[0]:4d} nodes  | feature_dim={x.shape[1]}")
  print(f"             sample IDs: {ids[:2]}")

print("\n🔗 EDGE TYPES:")
for (src, rel, dst) in sample.edge_types:
  ei = sample[src, rel, dst].edge_index
  print(f"  ({src}) --[{rel}]--> ({dst})")
  print(f"             shape={ei.shape}  ({ei.shape[1]} edges)")

print("\n📊 POST ENGAGEMENT FEATURES:")
eng = sample["post"].engagement
labels = ["like","love","haha","wow","sad","angry","care","comments","shares","total_reactions"]
for l, v in zip(labels, eng[0].tolist()):
  if v > 0:
      print(f"  {l}: {int(v)}")

print("\n📝 RAW TEXT (from JSON):")
raw = json.loads(open(json_files[0]).read())
print(f"  {repr((raw['node_features'].get('text') or '')[:100])}")

print("\n💬 COMMENT TREE (top 2):")
tree = raw["graph_structure"].get("comment_tree", [])
for c in tree[:2]:
  print(f"  [{c.get('author_name')}]: {repr((c.get('raw_text') or '')[:60])}")
  for r in c.get("replies", [])[:1]:
      print(f"    ↳ [{r.get('author_name')}]: {repr((r.get('raw_text') or '')[:50])}")

print(f"\n  Total comments: {sum(1 + len(c.get('replies',[])) for c in tree)}")
print(f"  Total edges:    {sum(sample[e].edge_index.shape[1] for e in sample.edge_types)}")

HETERODATA STRUCTURE

📦 NODE TYPES:
  post       →    1 nodes  | feature_dim=1
             sample IDs: ['pfbid012rtany3vdn2YWAdtXfPtow8tLwo3agWrUrxK8bEz5nrw8hXuR5BnjdUvnD7ZxTrl']
  user       →  155 nodes  | feature_dim=1
             sample IDs: ['thethao247.vn', 'hoang.x.son.7']
  comment    →  186 nodes  | feature_dim=1
             sample IDs: ['cmt_6d78c9d3', 'cmt_7e4c44d2']
  hashtag    →    0 nodes  | feature_dim=1
             sample IDs: []

🔗 EDGE TYPES:
  (user) --[author]--> (post)
             shape=torch.Size([2, 1])  (1 edges)
  (user) --[author]--> (comment)
             shape=torch.Size([2, 186])  (186 edges)
  (user) --[mention]--> (user)
             shape=torch.Size([2, 4])  (4 edges)
  (user) --[reply]--> (user)
             shape=torch.Size([2, 89])  (89 edges)
  (user) --[mention_rev]--> (user)
             shape=torch.Size([2, 4])  (4 edges)
  (user) --[reply_rev]--> (user)
             shape=torch.Size([2, 89])  (89 edges)
  (comment) --[reply_to]--> (post)
  

In [9]:
from PIL import Image
import torch
from transformers import AutoModel, AutoTokenizer, CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

text_tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")
text_model = AutoModel.from_pretrained("uitnlp/visobert").to(device)
text_model.eval()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

@torch.no_grad()
def get_text_embedding(texts, batch_size=32):
    if not texts: return torch.empty((0, 768))
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = text_tokenizer(batch, padding=True, truncation=True,
                                max_length=128, return_tensors="pt").to(device)
        out = text_model(**inputs)
        all_embs.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(all_embs, dim=0)

@torch.no_grad()
def get_image_embedding(image_paths, batch_size=16):
    """Load real images from disk, embed via CLIP. Fallback zeros if missing."""
    if not image_paths:
        return torch.zeros((1, 512))
    imgs = []
    for p in image_paths:
        full_path = p.replace("data/media/", f"{MEDIA_DIR}/")
        try:
            imgs.append(Image.open(full_path).convert("RGB"))
        except:
            imgs.append(Image.new("RGB", (224, 224)))
    all_embs = []
    for i in range(0, len(imgs), batch_size):
        batch_imgs = imgs[i:i+batch_size]
        inputs = clip_processor(images=batch_imgs, return_tensors="pt").to(device)
        emb = clip_model.get_image_features(**inputs)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0).mean(dim=0, keepdim=True)

print("✅ Models loaded")

Device: cuda


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

✅ Models loaded


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch_geometric.nn import HGTConv
from torch_geometric.utils import softmax, degree
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter as pyg_scatter
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# -- Helper: resolve image path in Kaggle -----------------------------------------
def resolve_img_path(local_path):
    if not local_path:
        return None
    suffix = local_path.split('data/media/')[-1] if 'data/media/' in local_path else local_path
    candidates = [
        f'{MEDIA_DIR}/{suffix}',
        f'/kaggle/input/facebook-graph/data/media/{suffix}',
        f'/kaggle/input/datasets/taanhkhoa/facebook-graph/data/media/{suffix}',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

@torch.no_grad()
def embed_image(path):
    """Load 1 image from disk -> CLIP embedding [1, 512]. Zeros if missing."""
    full = resolve_img_path(path) if path else None
    try:
        img = Image.open(full).convert('RGB')
        inputs = clip_processor(images=[img], return_tensors='pt').to(device)
        return clip_model.get_image_features(**inputs).cpu()  # [1, 512]
    except Exception:
        return torch.zeros(1, 512)

# -- Build processed graphs -------------------------------------------------------
# For old posts (no image_nodes in graph_structure), we synthesise image nodes
# from node_features.local_images and add (post, contains, image) edges.
# For new posts (image_nodes populated), we use the stored data directly.

processed_graphs = []
all_node_types = set()
all_edge_types = set()

TEXT_DIM, IMG_DIM = 768, 512

print('Processing graphs...')
for i in tqdm(range(len(dataset))):
    data = dataset[i]
    raw = json.loads(open(json_files[i]).read())
    gs  = raw.get('graph_structure', {})
    nf  = raw.get('node_features', {})

    # -- Post text embedding --
    post_text = nf.get('text') or 'Khong co noi dung'
    data['post'].x_text = get_text_embedding([post_text]).cpu()  # [1, 768]

    # -- Post engagement features --
    eng = raw.get('engagement', {})
    data['post'].x_eng = torch.log1p(torch.tensor([[
        eng.get('like', 0),           eng.get('love', 0),
        eng.get('haha', 0),           eng.get('wow', 0),
        eng.get('sad', 0),            eng.get('angry', 0),
        eng.get('care', 0),           eng.get('comment_count', 0),
        eng.get('share_count', 0),    eng.get('total_reactions', 0),
    ]], dtype=torch.float32))  # [1, 10] — log1p compresses large counts

    # -- Comment text + per-comment image embeddings --
    cmt_map     = {}  # comment_id -> raw_text
    cmt_img_map = {}  # comment_id -> first local_image_path
    def walk(cmts):
        for c in cmts:
            cmt_map[c['comment_id']] = c.get('raw_text') or 'Khong co noi dung'
            paths = [p for p in c.get('local_image_paths', []) if p]
            if paths:
                cmt_img_map[c['comment_id']] = paths[0]
            walk(c.get('replies', []))
    walk(gs.get('comment_tree', []))

    comment_ids   = data['comment'].node_id
    comment_texts = [cmt_map.get(cid, 'Khong co noi dung') for cid in comment_ids]
    if comment_texts:
        text_embs    = get_text_embedding(comment_texts).cpu()        # [N, 768]
        cmt_img_embs = torch.cat([
            embed_image(cmt_img_map.get(cid)) if cid in cmt_img_map
            else torch.zeros(1, IMG_DIM)
            for cid in comment_ids
        ], dim=0)                                                      # [N, 512]
        data['comment'].x = torch.cat([text_embs, cmt_img_embs], dim=-1)  # [N, 1280]
    else:
        data['comment'].x = torch.zeros(0, TEXT_DIM + IMG_DIM)

    # -- Image nodes --
    # NEW format: image_nodes list is populated in graph_structure.
    # OLD format (existing ~920 posts): image_nodes is absent/empty; fall back to
    #   node_features.local_images and synthesise image nodes + edges manually.
    post_id       = raw['post_id']
    img_node_list = gs.get('image_nodes', [])
    local_images  = nf.get('local_images', [])

    if img_node_list:
        # New format -- image nodes and edges already encoded in HeteroData by
        # to_pyg_heterodata (via edges_content_image).  Just embed features.
        img_embs = torch.cat(
            [embed_image(n.get('local_path', '')) for n in img_node_list],
            dim=0
        )  # [num_images, 512]
        data['image'].x = img_embs

    elif local_images:
        # Old format -- synthesise image nodes from local_images list.
        # Create one image node per path and wire:
        #   (post, contains, image)     row=post_idx, col=img_idx
        #   (image, contained_by, post) row=img_idx,  col=post_idx
        num_imgs = len(local_images)
        img_embs = torch.cat(
            [embed_image(p) for p in local_images],
            dim=0
        )  # [num_imgs, 512]
        data['image'].x       = img_embs
        data['image'].node_id = [
            f'{post_id}_img_{k:03d}' for k in range(num_imgs)
        ]

        post_idx    = 0  # single post per graph, always index 0
        img_indices  = torch.arange(num_imgs, dtype=torch.long)
        post_indices = torch.zeros(num_imgs, dtype=torch.long)  # all point to post 0

        data['post',  'contains',     'image'].edge_index = torch.stack(
            [post_indices, img_indices], dim=0)   # [2, num_imgs]
        data['image', 'contained_by', 'post' ].edge_index = torch.stack(
            [img_indices, post_indices], dim=0)   # [2, num_imgs]

    else:
        # No images at all — must initialise empty edge tensors so PyG
        # DataLoader can collate this graph with image-bearing graphs.
        data['image'].x       = torch.zeros(0, IMG_DIM)
        data['image'].node_id = []
        data['post',  'contains',     'image'].edge_index = torch.empty((2, 0), dtype=torch.long)
        data['image', 'contained_by', 'post' ].edge_index = torch.empty((2, 0), dtype=torch.long)

    # -- User structural features --
    num_users = data['user'].num_nodes
    data['user'].x = torch.zeros(num_users, 3)
    if num_users > 0:
        data['user'].x[0, 0] = 1.0  # index 0 = page/post author

    # -- Remove sparse hashtag nodes (not used in this model) --
    if 'hashtag' in data.node_types:
        del data['hashtag']
    if ('post', 'has_hashtag', 'hashtag') in data.edge_types:
        del data['post', 'has_hashtag', 'hashtag']

    all_node_types.update(data.node_types)
    all_edge_types.update(data.edge_types)
    processed_graphs.append(data)

global_metadata = (list(all_node_types), list(all_edge_types))
n_with_img = sum(1 for g in processed_graphs if g['image'].num_nodes > 0)
print(f'\nDone. Node types: {global_metadata[0]}')
print(f'   Posts with image nodes: {n_with_img}/{len(processed_graphs)}')


# -- Model -----------------------------------------------------------------------
class SocialMultimodalHGT(nn.Module):
    """
    Heterogeneous Graph Transformer for Vietnamese social-media posts.

    Node input dimensions:
      post    : x_text [N, 768]  + x_eng [N, 10]
      comment : x      [N, 1280] (text 768 + image 512)
      user    : x      [N, 3]    (structural flags)
      image   : x      [N, 512]  (CLIP embedding)
    """
    def __init__(self, hidden_dim: int, metadata):
        super().__init__()
        # Post projections
        self.post_text_proj = nn.Linear(768, hidden_dim)
        self.post_eng_proj  = nn.Linear(10,  hidden_dim // 4)
        self.post_fuse      = nn.Linear(hidden_dim + hidden_dim // 4, hidden_dim)
        # Comment projection: text(768) + image(512) = 1280
        self.comment_proj   = nn.Linear(768 + 512, hidden_dim)
        # User structural projection
        self.user_proj      = nn.Linear(3, hidden_dim)
        # Image projection: CLIP 512 -> hidden_dim
        self.image_proj     = nn.Linear(512, hidden_dim)

        self.hgt      = HGTConv(hidden_dim, hidden_dim, metadata=metadata, heads=4)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        p_text = F.gelu(self.post_text_proj(x_dict['post_text']))  # [B, hidden]
        p_eng  = F.gelu(self.post_eng_proj(x_dict['post_eng']))    # [B, hidden//4]
        h_dict = {
            'post':    F.gelu(self.post_fuse(torch.cat([p_text, p_eng], dim=-1))),
            'comment': F.gelu(self.comment_proj(x_dict['comment'])),
            'user':    F.gelu(self.user_proj(x_dict['user'])),
            'image':   F.gelu(self.image_proj(x_dict['image'])),
        }
        out = self.hgt(h_dict, edge_index_dict)
        return {k: self.out_proj(v) for k, v in out.items()}


# -- Loss: topology-guided image-comment alignment --------------------------------
def topology_guided_alignment_loss(out_dict, batch, device):
    """
    NT-Xent contrastive loss between per-post image repr and per-post comment repr.

    For each post p in the batch:
      image_repr[p]   = mean of image node embeddings linked to p via
                        (post, contains, image) edges.
      comment_repr[p] = topology-weighted mean of comment embeddings linked
                        to p via (comment, reply_to, post) edges, where
                        weight = log(1 + in-degree) normalised per post.
                        In-degree of a comment = how many other comments
                        replied to it via (comment, reply_to, comment) edges,
                        making it a measure of conversational centrality.

    Only posts that have at least one image AND at least one comment contribute
    to the loss; posts with zero image nodes are skipped.
    """
    img_out     = out_dict.get('image')    # [total_imgs_in_batch, hidden]
    comment_out = out_dict.get('comment')  # [total_cmts_in_batch, hidden]
    post_out    = out_dict.get('post')     # [total_posts_in_batch, hidden]

    if (img_out is None or comment_out is None
            or img_out.size(0) == 0 or comment_out.size(0) == 0):
        dummy = torch.zeros(1, requires_grad=True, device=device)
        return dummy.squeeze(), torch.zeros(1, 1, device=device)

    num_posts = post_out.size(0)

    # -- 1. Aggregate image embeddings per post -----------------------------------
    # Edge: (post, contains, image)  shape [2, E_pi]  row=post_idx, col=img_idx
    if ('post', 'contains', 'image') not in batch.edge_types:
        dummy = torch.zeros(1, requires_grad=True, device=device)
        return dummy.squeeze(), torch.zeros(1, 1, device=device)

    ei_pi       = batch['post', 'contains', 'image'].edge_index  # [2, E_pi]
    post_of_img = ei_pi[0]   # which post each image belongs to
    img_idx_    = ei_pi[1]   # which image node

    # Mean-pool image embeddings per post: result shape [num_posts, hidden]
    img_per_post = pyg_scatter(
        img_out[img_idx_],          # [E_pi, hidden]
        post_of_img,                # scatter index
        dim=0,
        dim_size=num_posts,
        reduce='mean',
    )  # [num_posts, hidden]

    # -- 2. Aggregate comment embeddings per post ---------------------------------
    # Edge: (comment, reply_to, post)  row=cmt_idx, col=post_idx
    if ('comment', 'reply_to', 'post') not in batch.edge_types:
        dummy = torch.zeros(1, requires_grad=True, device=device)
        return dummy.squeeze(), torch.zeros(1, 1, device=device)

    ei_cp       = batch['comment', 'reply_to', 'post'].edge_index  # [2, E_cp]
    cmt_idx_    = ei_cp[0]   # source: comment
    post_of_cmt = ei_cp[1]   # target: post

    # Topology weight: in-degree of each comment in (comment, reply_to, comment).
    # Edge convention: row=reply, col=parent.  degree(col) = in-degree of parent,
    # i.e. how many replies the parent comment received.  More replies = more
    # central in the conversation => higher weight when aggregating to the post.
    num_comments = comment_out.size(0)
    if ('comment', 'reply_to', 'comment') in batch.edge_types:
        ei_cc  = batch['comment', 'reply_to', 'comment'].edge_index  # [2, E_cc]
        in_deg = degree(ei_cc[1], num_nodes=num_comments, dtype=torch.float)
    else:
        in_deg = torch.zeros(num_comments, device=device)

    # Per-edge weight = log(1 + in_degree) of the source comment
    edge_weights = torch.log1p(in_deg[cmt_idx_])  # [E_cp]

    # Normalise weights within each target post (softmax over edges per post)
    alpha = softmax(edge_weights, index=post_of_cmt, num_nodes=num_posts)  # [E_cp]

    # Weighted scatter: comment_repr[p] = sum_i alpha_i * comment_out[i]
    cmt_per_post = pyg_scatter(
        comment_out[cmt_idx_] * alpha.unsqueeze(-1),  # [E_cp, hidden]
        post_of_cmt,
        dim=0,
        dim_size=num_posts,
        reduce='sum',
    )  # [num_posts, hidden]

    # -- 3. Select posts that have BOTH image and comment contributions -----------
    img_norm_sq = img_per_post.norm(dim=-1)   # 0 for posts with no images
    cmt_norm_sq = cmt_per_post.norm(dim=-1)   # 0 for posts with no comments
    valid_mask  = (img_norm_sq > 1e-6) & (cmt_norm_sq > 1e-6)

    if valid_mask.sum() < 2:
        # Not enough paired posts for a contrastive loss -- return 0
        dummy = torch.zeros(1, requires_grad=True, device=device)
        return dummy.squeeze(), torch.zeros(1, 1, device=device)

    img_repr = img_per_post[valid_mask]   # [P, hidden]
    cmt_repr = cmt_per_post[valid_mask]   # [P, hidden]

    # -- 4. NT-Xent loss ---------------------------------------------------------
    img_norm = F.normalize(img_repr, p=2, dim=-1)   # [P, hidden]
    cmt_norm = F.normalize(cmt_repr, p=2, dim=-1)   # [P, hidden]

    logits = torch.matmul(img_norm, cmt_norm.t()) / 0.07  # [P, P]
    labels = torch.arange(img_repr.size(0), device=device)
    loss   = F.cross_entropy(logits, labels)
    return loss, logits


# -- Training loop ---------------------------------------------------------------
train_loader = DataLoader(processed_graphs[:600], batch_size=16, shuffle=True)
model        = SocialMultimodalHGT(hidden_dim=256, metadata=global_metadata).to(device)
optimizer    = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)

print('\nTraining with Image-Comment Alignment...')
model.train()
for epoch in range(15):
    total_loss = correct = total = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        x_dict = {
            'post_text': batch['post'].x_text,
            'post_eng':  batch['post'].x_eng,
            'comment':   batch['comment'].x,
            'user':      batch['user'].x,
            'image':     batch['image'].x,
        }
        out_dict = model(x_dict, batch.edge_index_dict)

        loss, logits = topology_guided_alignment_loss(out_dict, batch, device)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if logits.shape[0] > 1:
            preds   = logits.argmax(dim=1)
            labels_ = torch.arange(logits.size(0), device=device)
            correct += (preds == labels_).sum().item()
            total   += logits.size(0)

    print(f'Epoch {epoch+1:02d}/15 | '
          f'Loss: {total_loss / len(train_loader):.4f} | '
          f'Acc: {correct / max(total, 1) * 100:.1f}%')


In [11]:
# ── Sample Inspection: 1 post đầy đủ ────────────────────────────────────────
import json, glob, os
from pathlib import Path

# Lấy post có đủ: text + ảnh + comments có ảnh
def find_rich_sample(json_files, top_n=5):
    candidates = []
    for fp in json_files:
        d = json.loads(open(fp).read())
        nf = d.get("node_features", {})
        def all_cmts(cmts):
            r = []
            for c in cmts: r.append(c); r.extend(all_cmts(c.get("replies", [])))
            return r
        cmts = all_cmts(d["graph_structure"].get("comment_tree", []))
        score = (len(nf.get("local_images", [])) * 10
                 + len(cmts)
                 + sum(1 for c in cmts if c.get("image_urls")) * 5
                 + (5 if nf.get("text") else 0))
        candidates.append((score, fp, d))
    candidates.sort(reverse=True)
    return candidates[0][1], candidates[0][2]

_, d = find_rich_sample(json_files)

nf   = d["node_features"]
gs   = d["graph_structure"]
eng  = d["engagement"]
meta = d["metadata"]

def all_cmts(cmts):
    r = []
    for c in cmts: r.append((c, 0)); r.extend((x, depth+1) for x, depth in all_cmts_flat(c.get("replies", []), 1))
    return r

def all_cmts_flat(cmts, depth=0):
    r = []
    for c in cmts: r.append((c, depth)); r.extend(all_cmts_flat(c.get("replies", []), depth+1))
    return r

all_comments = all_cmts_flat(gs.get("comment_tree", []))

print("=" * 65)
print("📄 POST")
print("=" * 65)
print(f"  ID      : {d['post_id']}")
print(f"  URL     : {d['post_url']}")
print(f"  Page    : {gs.get('author_name')} ({gs.get('author_id')})")
print(f"  Type    : {meta.get('post_type', 'post')}")
print()

print("📝 TEXT:")
text = nf.get("text") or "(không có text)"
print(f"  {text[:300]}{'...' if len(text) > 300 else ''}")
print()

print("🖼️  IMAGES (post):")
local_imgs = nf.get("local_images", [])
if local_imgs:
    for p in local_imgs:
        full = p.replace("data/media/", f"{MEDIA_DIR}/")
        exists = os.path.exists(full)
        print(f"  {full}  [{'✅' if exists else '❌ missing'}]")
else:
    print("  (không có ảnh)")
print()

print("📊 ENGAGEMENT:")
print(f"  👍 {eng.get('like',0):,}  ❤️  {eng.get('love',0):,}  😂 {eng.get('haha',0):,}  "
      f"😮 {eng.get('wow',0):,}  😢 {eng.get('sad',0):,}  😡 {eng.get('angry',0):,}")
print(f"  💬 {eng.get('comment_count',0):,} comments  🔁 {eng.get('share_count',0):,} shares")
print()

print(f"💬 COMMENTS ({len(all_comments)} total)")
print("-" * 65)
depth_counts = {}
for _, d_ in all_comments:
    depth_counts[d_] = depth_counts.get(d_, 0) + 1
for depth, count in sorted(depth_counts.items()):
    label = ["Top-level", "Replies (depth 1)", "Sub-replies (depth 2+)"][min(depth, 2)]
    print(f"  Depth {depth} ({label}): {count}")
print()

# Show first 5 comments with details
print("📋 SAMPLE COMMENTS (first 5):")
for i, (c, depth) in enumerate(all_comments[:5]):
    indent = "  " + "  " * depth
    print(f"{indent}[Depth {depth}] {c.get('author_name', 'Unknown')}")
    txt = c.get("raw_text") or "(no text)"
    print(f"{indent}  💬 {txt[:80]}{'...' if len(txt) > 80 else ''}")
    print(f"{indent}  👍 {c.get('like_count', 0)} likes  | ts: {(c.get('timestamp') or 'N/A')[:19]}")
    img_urls = c.get("image_urls", [])
    local_img_paths = c.get("local_image_paths", [])
    if img_urls:
        print(f"{indent}  🖼️  {len(img_urls)} image(s)")
        for lp in local_img_paths[:1]:
            full = lp.replace("data/media/", f"{MEDIA_DIR}/")
            exists = os.path.exists(full)
            print(f"{indent}     → {full}  [{'✅' if exists else '❌'}]")
    print()

# Edge summary
print("🔗 GRAPH EDGES:")
edge_types = {
    "user→post (author)":      len(gs.get("edges_user_post", [])),
    "user→comment (author)":   len(gs.get("edges_user_comment", [])),
    "user→user (reply/mention)": len(gs.get("edges_user_user", [])),
    "comment→post/comment":    len(gs.get("edges_comment_reply", [])),
    "post→hashtag":            len(gs.get("edges_post_hashtag", [])),
}
for k, v in edge_types.items():
    print(f"  {k}: {v}")
print(f"  TOTAL: {sum(edge_types.values())}")

📄 POST
  ID      : pfbid02RoQZv41jECUELdZkwEydZKZ39X5RegstsYWdwzpRoW8vNBqK3kNw7NYXgaFsSfcRl
  URL     : https://www.facebook.com/PageWSS/posts/pfbid02RoQZv41jECUELdZkwEydZKZ39X5RegstsYWdwzpRoW8vNBqK3kNw7NYXgaFsSfcRl
  Page    : Why So Serious (PageWSS)
  Type    : post

📝 TEXT:
  Story mấy ngày nay kiểu

🖼️  IMAGES (post):
  /kaggle/input/datasets/taanhkhoa/facebook-graph/data/media/pfbid02RoQZv41jECUELdZkwEydZKZ39X5RegstsYWdwzpRoW8vNBqK3kNw7NYXgaFsSfcRl/img_000.jpg  [✅]

📊 ENGAGEMENT:
  👍 25,000  ❤️  0  😂 43,000  😮 0  😢 0  😡 0
  💬 578 comments  🔁 365 shares

💬 COMMENTS (460 total)
-----------------------------------------------------------------
  Depth 0 (Top-level): 401
  Depth 1 (Replies (depth 1)): 51
  Depth 2 (Sub-replies (depth 2+)): 8

📋 SAMPLE COMMENTS (first 5):
  [Depth 0] Giang Hoàng Nguyễn
    💬 kiểu tụi nó OVTK lâu năm á thông cảm 😂
    👍 0 likes  | ts: 2026-01-13T18:30:32

  [Depth 0] Ngọc Diễm
    💬 Y dị 🤣
    👍 0 likes  | ts: 2026-01-13T18:30:33
    🖼️  1 image(s)

  